# Permute pre-layer-1 activations

Compare clean, unrestricted within-segment permutation, and length-matched permutation. `PERMUTE_MODE` toggles between the middle segment and all segments.


In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
PERMUTE_MODE="middle"  # change to "all"
MIDDLE_SEGMENT_INDEX=1
PRE,POST="blocks.1.hook_resid_pre","blocks.1.hook_resid_post"
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=[PRE,POST]); clean_pre=c[PRE][0].detach().clone(); clean_post=c[POST][0].detach().cpu().numpy()
segments=line_segments(tokens,vocab); selected=set(range(len(segments))) if PERMUTE_MODE=="all" else {MIDDLE_SEGMENT_INDEX}; rng=np.random.default_rng(6009)
unrestricted=clean_pre.clone(); length_matched=clean_pre.clone()
lengths=np.array([token_length(t,vocab) for t in tokens])
for i,(start,end) in enumerate(segments):
 if i in selected and end-start>1:
  order=torch.as_tensor(rng.permutation(end-start),device=unrestricted.device); unrestricted[start:end]=clean_pre[start:end][order]
  for length in sorted(set(lengths[start:end])):
   if np.isnan(length): continue
   positions=np.flatnonzero(lengths[start:end]==length)+start
   if len(positions)>1:
    shuffled=positions[rng.permutation(len(positions))]
    length_matched[positions]=clean_pre[torch.as_tensor(shuffled,device=clean_pre.device)]
def run_post(replacement):
 capd={}
 def replace(x,hook): return replacement.unsqueeze(0).to(device=x.device,dtype=x.dtype)
 def cap(x,hook): capd["post"]=x.detach().cpu(); return x
 model.run_with_hooks(batch.tokens,prepend_bos=False,fwd_hooks=[(PRE,replace),(POST,cap)])
 return capd["post"][0].numpy()
unrestricted_post=run_post(unrestricted); length_matched_post=run_post(length_matched)
print("mode",PERMUTE_MODE,"selected",sorted(selected),"of",len(segments))
print("unrestricted pre MSE",float(torch.mean((unrestricted-clean_pre)**2)))
print("length-matched pre MSE",float(torch.mean((length_matched-clean_pre)**2)))
print("unrestricted post MSE",float(np.mean((unrestricted_post-clean_post)**2)))
print("length-matched post MSE",float(np.mean((length_matched_post-clean_post)**2)))
coords=shared_pca([clean_post,unrestricted_post,length_matched_post])
plot_trajectory(coords,["clean post layer 1",f"unrestricted permutation ({PERMUTE_MODE})",f"length-matched permutation ({PERMUTE_MODE})"],"Post-layer-1 trajectories after pre-layer-1 permutation","figures/layer1_pre_activation_permutation_comparison.png")


mode middle selected [1] of 8
unrestricted pre MSE 0.03131398558616638
length-matched pre MSE 0.014399035833775997
unrestricted post MSE 0.9103748798370361
length-matched post MSE 0.8386404514312744
